### Benchmarking for three different geocoders: 
-   degauss
-   nominatim
-   postgis Tiger

This code normalizes a set of addresses from the Miami-Dade Open Data hub and then provides a method for running the addresses through the three geocoders.

This notebook needs containerized versions of the geocoders running with avaialble APIs to request services from the containers. See the manifests for the containers and their services in the k8s directory. 

Absolutely no guarantee this will run, it has been a while. On the other hand it should give an idea of the approach used.

In [ ]:
import re
import pandas as pd
import psycopg2 as pg
from urllib.request import urlopen
from urllib.parse import urlencode
import simplejson
import csv
import json

In [ ]:
## 5000 mdc addresses
path = "./data/"
source_file = "mdc_5000_addresses.csv"
columns = {
    "id": "object_id",
    "address": "true_site_addr",
    "unit": "true_site_unit",
    "city": "true_site_city",
    "county": None,
    "postal": "true_site_zip_code",
    "state": None,
    "country": None
}
defaults = {
    "county": "miami-dade",
    "state": "florida",
    "country": "united states"
}

In [ ]:
addresses = pd.read_csv(path+source_file, delimiter=',',dtype={
    columns['id']: 'string',
    columns['postal']: 'string',
})

### add ordinal endings to all numeric street names

In [ ]:
# add ordinal endings to numbers
ordinal = lambda n: "%d%s" % (n,"tsnrhtdd"[(n//10%10!=1)*(n%10<4)*n%10::4])

# make address standard for all geocoders
def normalize_addr(addr):
    
    if not pd.isna(addr[columns['address']]):
    
        # remove all apartment or unit identifiers from street address
        street = addr[columns['address']].replace(",","").replace(".","")
        # if units are in the street address we need to remove them
        #if not pd.isna(addr[columns['unit']]):
        #    unit_length = len(addr[columns['unit']]) + 1
        #    if unit_length > 1:
        #        street = street[0:unit_length*-1]

        # replace 'Unincorporate County' with 'Miami' for city
        if addr[columns['city']] == 'Unincorporated County': addr[columns['address']] = 'Miami' 

        # add ordinal endings to street names in they are numeric
        pieces = street.split(' ')
        ignore = ['apt','unit','lot','#']
        ordinal_index, ignore_index = -1, -1
        if pieces[0] == '': pieces = pieces[1:]
        if len(pieces) == 0: return addr
        if not pieces[0].isdigit(): # not a numeric address??
            if re.search(r'(\d+)([A-Za-z]+)', pieces[0]):
                pieces = list(re.search(r'(\d+)([A-Za-z]+)', pieces[0]).groups()) + pieces[1:]
        if len(pieces) > 2:
            temp = []
            for index, piece in enumerate(pieces):
                if re.search(r'(\d+)([A-Za-z]+)', piece):
                    temp = temp + list(re.search(r'(\d+)([A-Za-z]+)', piece).groups())
                else:
                    temp = temp + [piece]
            pieces = temp
            for index, piece in enumerate(pieces):
                if index > 0:
                    if pieces[index].isnumeric(): 
                        pieces[index] = ordinal(int(pieces[index]))
                        # print(pieces[index])
                        if len(pieces) > (index + 1):
                            # print(pieces[index][-2:].lower(),pieces[index+1].lower())
                            if pieces[index][-2:].lower() == pieces[index+1].lower():
                                ordinal_index = index + 1
            if ordinal_index > 0: pieces.remove(pieces[ordinal_index])
            
            for index, piece in enumerate(pieces):
                for unit in ignore:
                    if piece.lower().startswith(unit): 
                        ignore_index = index
            if ignore_index > 0: pieces = pieces[:ignore_index]
                
            street = ' '.join(filter(None,pieces))
        addr[columns['address']] = street

        # remove trailing hyphens from zip
        addr[columns['postal']] = addr[columns['postal']].rstrip('-') if not pd.isna(addr[columns['postal']]) else ''
    
    # print (addr['Address'])
    return addr

In [ ]:
##
 # normalize all street addresses so all geocoders work
 ##
data_file = open(f"{path}{source_file[:-4]}_normalized2.csv", 'w', newline='')
csv_writer = csv.writer(data_file)
count = 0

##
 # loop through addresses and normalize
 ##
for index, address in addresses.iterrows(): #.iloc[0:10,:]
    addr = normalize_addr(address).to_dict()

    if count == 0:
        header = addr.keys()
        csv_writer.writerow(header)
        count += 1
    csv_writer.writerow(addr.values())
        
data_file.close()

In [ ]:
normalized_addresses = pd.read_csv(f"{path}{source_file[:-4]}.csv", delimiter=',', dtype={
    columns['id']: 'string',
    columns['postal']: 'string',
})

In [ ]:
normalized_addresses.iloc[0:30,:]

### Zip code lookup for counties improves results

In [ ]:
# from zip code tabulation areas provided by US census
# Including county names for Nominatim improves performance
zip_lookup = pd.read_csv('./data/zip_lookup.csv', delimiter=',',dtype={
    'Primary Official County Code': 'string',
    'Zip Code': 'string'
})

In [ ]:
zip_lookup.iloc[0:30,:]

### Degauss

In [ ]:
# degauss container through fastAPI, both on kubernetes
BASE_PATH = 'http://localhost:8000/geocode/?'
results = f"{path}degauss_{source_file}"
def construct_params(addr):
    return {
        "addr": f"{addr[columns['address']]}, {addr[columns['city']]}, {addr[columns['state']]}, {addr[columns['postal']]}"
    }

### Nominatim

In [ ]:
# nominatim conatiner running its own api on kubernetes
# using parametrized search ...
BASE_PATH = 'http://localhost:8088/search?'
results = path + "nominatim_" + source_file
def construct_params(addr):
    return {
        "street": addr[columns['address']],
        "city": addr[columns['city']],
        "county": addr[columns['county']],
        "state": addr[columns['state']],
        "country": addr[columns['country']],
        "postalcode": addr[columns['postal']]
    }

In [ ]:
# nominatim conatiner running its own api on kubernetes
# using open string search
BASE_PATH = 'http://localhost:8088/search?'
results = path + "nominatim_" + source_file
def construct_params(addr):
    if columns['city'] in addr:
        return {
            "q": (
                f"{addr[columns['address']]}, "
                f"{addr[columns['city']] if columns['city'] else defaults['city']}, "
                f"{addr[columns['state']] if columns['state'] else defaults['state']}, "
                f"{addr[columns['postal']]}"
            )
        }
    else:
        return {
            "q": (
                f"{addr[columns['address']]}, "
                f"{addr[columns['state']] if columns['state'] else defaults['state']}, "
                f"{addr[columns['postal']]}"
            )
        }

In [ ]:
# nominatim conatiner running its own api on kubernetes
# parametrized search checking for bad ZIPS
BASE_PATH = 'http://localhost:8088/search?'
results = path + "nominatim_" + source_file
def construct_params(addr):
    # add ordinal endings to numbers
    ordinal = lambda n: "%d%s" % (n,"tsnrhtdd"[(n//10%10!=1)*(n%10<4)*n%10::4])
    def add_ordinal(street):
        pieces = street.split(' ')
        if len(pieces) > 2:
            for index, piece in enumerate(pieces):
                if index > 0:
                    if pieces[index].isnumeric(): 
                        pieces[index] = ordinal(int(pieces[index]))
            street = ' '.join(pieces)
        return street
    county, state = 'Miami-Dade', 'Florida'
    if not pd.isna(addr[columns['postal']]):
        if len(zip_lookup[zip_lookup['Zip Code']==addr[columns['postal']][0:5]]) > 0:
            county = zip_lookup[zip_lookup['Zip Code']==addr[columns['postal']][0:5]]['Primary Official County Name'].values[0].lower()
            state = zip_lookup[zip_lookup['Zip Code']==addr[columns['postal']][0:5]]['Official State Name'].values[0].lower()
    return {
        "street": addr[columns['address']].lower(),
        "city": addr[columns['city']].lower(),
        "county": county,
        "state": state,
        "country": "united states",
        "postalcode": addr[columns['postal']]
    }

#### main loop for Degauss and Nominatim

In [ ]:
data_file.close()

In [ ]:
%%time

data_file = open(results, 'w', newline='',encoding="utf-8")
csv_writer = csv.writer(data_file)
count, fail, nope, errors = 0, 0, 0, 0
failed = {}
county = ''

##
 # loop through addresses and geocode
 ##
for index, address in normalized_addresses.iterrows(): # .iloc[0:10,:].iterrows() -or- .iterrows()

    if address[columns['address']] != 'nan' and not pd.isna(address[columns['address']]):
        query_parameters = construct_params(address)
        query_string  = urlencode(query_parameters)
    
        trying = True
        while trying:
            try:
                connection = urlopen("{}{}".format(BASE_PATH, query_string))
                response = simplejson.load(connection)
                if len(response) > 0:
                    response[0][columns['id']] = address[columns['id']]

                    if count == 0:
                        header = response[0].keys()
                        csv_writer.writerow(header)
                    count += 1
                    csv_writer.writerow(response[0].values())
                    trying = False
                    
                else:
                    print('\n',address,query_string,'\n')
                    if columns['city'] in address:
                        del address[columns['city']]
                        query_parameters = construct_params(address)
                        query_string  = urlencode(query_parameters)
                    else:
                        fail += 1
                        failed[fail] = address
                        trying = False
                        
            except Exception as error:
                print('error:',error)
                print("{}{}".format(BASE_PATH, query_string))
                errors += 1
                print(str(index) + '|' + str(count) + '|' + str(fail) + '|' + str(nope) + '|' + str(errors))
                
    else:
        nope += 1
        
    print ('\r'+ str(index) + '|' + str(count) + '|' + str(fail) + '|' + str(nope) + '|' + str(errors),end='')
    
print("\n")
data_file.close()

### PostGIS TIGER geocoder

In [ ]:
# PostGIS TIGER geocoder
conn = pg.connect("dbname=geocode user=postgres password=postgres")
results = 'postgis.csv'

In [ ]:
def construct_query(addr):
    return """SELECT g.rating, ST_X(g.geomout) As lon, ST_Y(g.geomout) As lat,
    (addy).address As stno, (addy).streetname As street,
    (addy).streettypeabbrev As styp, (addy).location As city, (addy).stateabbrev As state,(addy).zip
    FROM geocode('""" + addr['true_site_addr'] + ', ' + address['true_site_city'] + ' FL, ' + address['true_site_zip_code'] + "', 1) As g;"

In [ ]:
%%time

data_file = open(results, 'w', newline='')
csv_writer = csv.writer(data_file)
count = 0

for index, address in normalized_addresses.iterrows(): #.iloc[0:10,:]
    #print (address)
    if address['true_site_addr'] != 'nan' and not pd.isna(address['true_site_addr']) and address['source_country']:
        sql_query = construct_query(address)
        #print(sql_query)
        with conn.cursor() as curs:
            try:
                curs.execute(sql_query)
                response = [dict((curs.description[i][0], value) \
                   for i, value in enumerate(row)) for row in curs.fetchall()]
                response[0]['ogc_fid'] = address['ogc_fid']

                if count == 0:
                    header = response[0].keys()
                    csv_writer.writerow(header)
                count += 1
                csv_writer.writerow(response[0].values())
                #print(response)

            # a more robust way of handling errors
            except (Exception, pg.DatabaseError) as error:
                print(error)

    print('\r' + str(count), end='')
    
data_file.close()

In [ ]:
conn.close()